In [4]:
from config_012 import *
from subpool_parsing_utils_012 import *
from subpool_plotting_utils_012 import *
from subpool_pdf_report_012 import *

import pandas as pd
import os


In [5]:
config_df = pd.read_csv(config_path, sep='\t')
subpools = get_subpools(config_df, plate)

In [6]:
sample_df = parse_sample_df(sample_metadata_path)
sample_df = sample_df[sample_df['plate'] == plate]

# Determine color mapping
color_by = 'Tissue' if sample_df['Tissue'].nunique() > sample_df['Genotype'].nunique() else 'genotype'
color_dict = get_color_dict(sample_df, color_by)

# Plot plate map
plate_map_path = plot_plate_map(sample_df, color_by, color_dict)


In [7]:
for subpool in subpools:

    # Alignment stats
    align_df = load_alignment_stats(plate, subpool)

    # QC summary tables
    qc_200umi, qc_500umi = load_qc_stats(plate, subpool, sampletype)

    # Cell counts
    formatted_counts = summarize_cell_counts(qc_200umi, qc_500umi, align_df, sampletype)

    # Generate knee plot
    knee_plot_path = plot_knee_raw_counts(plate, subpool)

    # Load obs table
    adata_obs = pd.read_csv(obs_path)
    adata_obs = adata_obs[adata_obs['subpool'] == subpool]

    # Heatmaps
    hmap_paths = [create_round_heatmap(adata_obs, round_col, kit, sampletype) for round_col in ['bc1_well', 'bc2_well', 'bc3_well']]

    # Cellbender
    cb_results_df = load_cellbender_stats(plate, subpool, sampletype)
    cb_settings_df = load_cellbender_settings(plate, subpool)
    cb_knee_path = plot_knee_cb(adata_obs)

    # Violin plots
    violin1_path, violin2_path = plot_qc_violins(adata_obs, plate)
    adata_obs_filt = filter_obs(adata_obs, min_counts, max_counts, min_genes, pct_counts_mt, doublet_score)
    violin_filt1, violin_filt2 = plot_qc_violins_filtered(adata_obs_filt, plate)

    # Barcode map
    barcode_map_df = create_barcode_sample_map(plate, kit, chemistry)

#     main_celltype_path = plot_stacked_main(combined_obs, subpool, main_tiss, "plots/sample_celltype_proportions_main_subpool.png")
#     mult_celltype_path = plot_stacked_mult(combined_obs, subpool, multi_tiss, "plots/sample_celltype_proportions_mult_subpool.png")

    # Create report
    elements = build_pdf_report(
        plate,
        config_df,
        kit,
        chemistry,
        sample_df,
        subpool, # SUBPOOL
        color_by,
        plate_map_path,
        align_df,
        formatted_counts,
        qc_200umi,
        qc_500umi,
        knee_plot_path,
        hmap_paths,
        cb_settings_df,
        cb_results_df,
        cb_knee_path, 
        violin1_path,
        violin2_path,
        violin_filt1,
        violin_filt2,
        #main_celltype_path,
        #mult_celltype_path,
        barcode_map_df,
        adata_obs,
        adata_obs_filt,
        #combined_obs,
        #main_tiss,
        #multi_tiss,
        sampletype,
        min_counts, 
        min_genes, 
        max_counts, 
        pct_counts_mt, 
        doublet_score
    )

    # Save PDF
    doc = create_pdf_doc(plate, subpool)
    doc.build(elements)
    print(f"Report saved as {doc.filename}")



Report saved as igvf_012/igvf_012_Subpool_1_report.pdf
Report saved as igvf_012/igvf_012_Subpool_2_report.pdf
Report saved as igvf_012/igvf_012_Subpool_3_report.pdf
Report saved as igvf_012/igvf_012_Subpool_4_report.pdf
Report saved as igvf_012/igvf_012_Subpool_5_report.pdf
Report saved as igvf_012/igvf_012_Subpool_6_report.pdf
Report saved as igvf_012/igvf_012_Subpool_7_report.pdf
Report saved as igvf_012/igvf_012_Subpool_8_report.pdf
